# Financial Econometrics — Homework 1

**Simulation practice: simple OLS regression**

The population model is

$$
y_i = \beta_0 + \beta_1 x_i + \varepsilon_i, \qquad i = 1, \ldots, n,
$$

with true parameters $\beta_0 = 5$ and $\beta_1 = -2.5$. In matrix form,

$$
y = X\beta + \varepsilon, \qquad X = [\,1 \;\; x\,], \qquad
\hat\beta = (X'X)^{-1} X'y.
$$

The regressor is $x_i \sim U(0, 1)$, independent of the error $\varepsilon_i \sim N(0, V)$.

| DGP | $V$ | $n$ |
| --- | --- | --- |
| DGP1 | 0.16 | 20 |
| DGP2 | 1 | 20 |
| DGP3 | 1 | 200 |

Each design is drawn 10 times from one seeded generator. Run 1 of each design is the sample that is plotted and reported as a single regression. The ten estimates are then averaged.

Launch this notebook from the project root so figures are written to `figures/`:

```bash
uv run jupyter notebook homework1.ipynb
```


## 1. Setup

`SEED` fixes the stream of random numbers, so restarting the kernel and running every cell from the top reproduces the same samples.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

SEED = 2026
rng = np.random.default_rng(SEED)

BETA0, BETA1 = 5.0, -2.5
N_RUNS = 10

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

np.set_printoptions(precision=4, suppress=True)


## 2. Draw one sample

$x$ is an $n \times 1$ vector of uniform draws. The error is an $n \times 1$ normal vector with variance $V$, and

$$
y = \beta_0 + \beta_1 x + \varepsilon.
$$


In [2]:
def simulate(n, V, generator=None):
    """Return one sample as column vectors y, x, eps, each n x 1."""
    generator = rng if generator is None else generator
    x = generator.uniform(0, 1, size=(n, 1))
    eps = generator.normal(0, np.sqrt(V), size=(n, 1))
    y = BETA0 + BETA1 * x + eps
    return y, x, eps

## 3. OLS estimator

Stack a column of ones next to $x$ and compute the normal-equation estimator

$$
\hat\beta = (X'X)^{-1} X'y.
$$


In [3]:
def ols(y, x):
    """OLS via beta_hat = (X'X)^{-1} X'y, with X = [1 x]."""
    ones = np.ones_like(x)
    X = np.hstack([ones, x])
    beta_hat = np.linalg.inv(X.T @ X) @ (X.T @ y)
    return X, beta_hat


## 4. Plot and repeat

The figure for a single sample shows the scatter of $y_i$ against $x_i$, the fitted line $\hat y = \hat\beta_0 + \hat\beta_1 x$, and the true line $y = 5 - 2.5x$ for reference.

Each DGP is then simulated 10 times and the ten estimates are averaged.

In [4]:
def plot_fit(name, n, V, x, y, beta_hat):
    """Scatter, OLS fit, and the true line for the first replication."""
    fig, ax = plt.subplots(figsize=(5.5, 4))
    ax.scatter(x, y, s=18, alpha=0.7, label="Simulated data")
    xs = np.linspace(0, 1, 100)
    b0, b1 = beta_hat[0, 0], beta_hat[1, 0]
    ax.plot(
        xs,
        b0 + b1 * xs,
        color="crimson",
        lw=2,
        label=rf"OLS fit: $\hat y={b0:.3f}{b1:+.3f}x$",
    )
    ax.plot(
        xs,
        BETA0 + BETA1 * xs,
        color="black",
        ls="--",
        lw=1,
        label=r"True line: $y=5-2.5x$",
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"{name}: V = {V}, n = {n} (run 1)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    path = FIG_DIR / f"{name.lower()}_fit.png"
    fig.savefig(path, dpi=200)
    print(f"Figure saved: {path}")
    plt.show()


def summarize(name, estimates):
    """Print the 10 estimates and their average."""
    mean = estimates.mean(axis=0)
    print(f"=== {name} ===")
    print("Run   beta0_hat   beta1_hat")
    for r, (b0, b1) in enumerate(estimates, 1):
        print(f"{r:>3d}   {b0:9.4f}   {b1:9.4f}")
    print(f"Mean  {mean[0]:9.4f}   {mean[1]:9.4f}   true = ({BETA0:.1f}, {BETA1:.1f})")
    sd = estimates.std(axis=0, ddof=1)
    print(f"SD    {sd[0]:9.4f}   {sd[1]:9.4f} (sample SD across runs)")
    print("Monte Carlo SE of mean:", sd / np.sqrt(len(estimates)))


def run_dgp(name, n, V, n_runs=N_RUNS):
    """Repeat the DGP n_runs times. Plot and report replication 1."""
    estimates = np.zeros((n_runs, 2))
    for r in range(n_runs):
        y, x, eps = simulate(n, V)
        _, beta_hat = ols(y, x)
        estimates[r] = beta_hat.ravel()
        if r == 0:
            print(f"{name}: stored y, x, eps with shapes {y.shape}, {x.shape}, {eps.shape}")
            print(
                f"Run 1 estimate: beta0_hat = {beta_hat[0, 0]:.4f}, "
                f"beta1_hat = {beta_hat[1, 0]:.4f}"
            )
            plot_fit(name, n, V, x, y, beta_hat)
    return estimates

## 5. DGP1 — $V = 0.16$, $n = 20$

Low noise, small sample. The scatter should sit close to the true line, and one OLS fit should already be near $(5, -2.5)$.


In [5]:
dgp1 = run_dgp("DGP1", n=20, V=0.16)
summarize("DGP1", dgp1)

DGP1: stored y, x, eps with shapes (20, 1), (20, 1), (20, 1)
Run 1 estimate: beta0_hat = 5.0865, beta1_hat = -2.5045
Figure saved: figures/dgp1_fit.png
=== DGP1 ===
Run   beta0_hat   beta1_hat
  1      5.0865     -2.5045
  2      4.7899     -2.2518
  3      4.9072     -2.4293
  4      4.7564     -2.1248
  5      4.9386     -2.3473
  6      4.6632     -2.1162
  7      5.1338     -2.8707
  8      4.9058     -2.0711
  9      4.6653     -2.1443
 10      4.9624     -2.2975
Mean     4.8809     -2.3158   true = (5.0, -2.5)
SD       0.1616      0.2421 (sample SD across runs)
Monte Carlo SE of mean: [0.0511 0.0766]


## 6. DGP2 — $V = 1$, $n = 20$

Same sample size as DGP1, with error variance $1 / 0.16 = 6.25$ times larger. The cloud of points is wider, so a single fit can tilt away from the true line, and the ten estimates should be more dispersed.


In [6]:
dgp2 = run_dgp("DGP2", n=20, V=1.00)
summarize("DGP2", dgp2)

DGP2: stored y, x, eps with shapes (20, 1), (20, 1), (20, 1)
Run 1 estimate: beta0_hat = 5.7230, beta1_hat = -2.9348
Figure saved: figures/dgp2_fit.png
=== DGP2 ===
Run   beta0_hat   beta1_hat
  1      5.7230     -2.9348
  2      4.6799     -2.2719
  3      5.0915     -2.5309
  4      5.0644     -3.1262
  5      4.9944     -3.0375
  6      5.2115     -2.8415
  7      4.8987     -2.8717
  8      4.6316     -1.7000
  9      5.6359     -3.7015
 10      5.2500     -2.8237
Mean     5.1181     -2.7840   true = (5.0, -2.5)
SD       0.3584      0.5333 (sample SD across runs)
Monte Carlo SE of mean: [0.1133 0.1687]


## 7. DGP3 — $V = 1$, $n = 200$

Same noise as DGP2, ten times as many observations. The scatter is still noisy, but the fitted line should lie close to the true line.

In [7]:
dgp3 = run_dgp("DGP3", n=200, V=1.00)
summarize("DGP3", dgp3)

DGP3: stored y, x, eps with shapes (200, 1), (200, 1), (200, 1)
Run 1 estimate: beta0_hat = 4.8357, beta1_hat = -2.4485
Figure saved: figures/dgp3_fit.png
=== DGP3 ===
Run   beta0_hat   beta1_hat
  1      4.8357     -2.4485
  2      4.6798     -2.1085
  3      4.7245     -2.2813
  4      5.1651     -2.8934
  5      5.0535     -2.6810
  6      4.8758     -2.3414
  7      5.0708     -2.5751
  8      5.4388     -3.1286
  9      4.7588     -2.1055
 10      5.0194     -2.3073
Mean     4.9622     -2.4871   true = (5.0, -2.5)
SD       0.2340      0.3347 (sample SD across runs)
Monte Carlo SE of mean: [0.074  0.1058]


## 8. Comparison across designs

The table collects the average of the ten estimates of $\hat\beta$ for each DGP.

In [8]:
def comparison_row(name, estimates, n, V):
    """Print one row of the comparison table."""
    mean = estimates.mean(axis=0)
    print(f"{name:<6} {V:6.2f} {n:5d} {mean[0]:10.4f} {mean[1]:10.4f}")


header = f"{'DGP':<6} {'V':>6} {'n':>5} {'mean b0':>10} {'mean b1':>10}"
print(header)
print("-" * len(header))
comparison_row("DGP1", dgp1, n=20, V=0.16)
comparison_row("DGP2", dgp2, n=20, V=1.00)
comparison_row("DGP3", dgp3, n=200, V=1.00)
print(f"{'True':<6} {'':>6} {'':>5} {BETA0:10.4f} {BETA1:10.4f}")

DGP         V     n    mean b0    mean b1
-----------------------------------------
DGP1     0.16    20     4.8809    -2.3158
DGP2     1.00    20     5.1181    -2.7840
DGP3     1.00   200     4.9622    -2.4871
True                    5.0000    -2.5000


## 9. What the designs show

The averages of the ten estimates are $(4.881, -2.316)$ for DGP1, $(5.118, -2.784)$ for DGP2, and $(4.962, -2.487)$ for DGP3, against the true $(5, -2.5)$. Ten replications cannot establish unbiasedness empirically. OLS is unbiased here because $x$ is independent of $\varepsilon$ and the error has mean zero, so $E[\hat\beta] = \beta$. With only ten runs the averages still move around the true values. DGP3, the large sample, lands closest.

**DGP1 versus DGP2** ($n$ fixed at 20, $V$ from 0.16 to 1). Raising the error variance spreads $y$ around the same line. The DGP1 scatter sits close to the true line, and run 1 gives $(5.087, -2.505)$. The DGP2 scatter is a wide cloud, and run 1 gives $(5.723, -2.935)$. Across the ten runs the DGP1 slopes stay between $-2.87$ and $-2.07$, while the DGP2 slopes range from $-3.70$ to $-1.70$.

**DGP2 versus DGP3** ($V$ fixed at 1, $n$ from 20 to 200). The points are as noisy as in DGP2, but the fitted line for run 1 is $(4.836, -2.448)$, almost on the true line. The ten DGP3 slopes range from $-3.13$ to $-2.11$, about half the range of DGP2, and the average is the closest of the three to $(5, -2.5)$.

**Why.** The estimation error is $\hat\beta - \beta = (X'X)^{-1}X'\varepsilon$, a weighted sum of the errors, with $\mathrm{Var}(\hat\beta \mid X) = V(X'X)^{-1}$. A larger $V$ increases the error variance and, holding $X$ fixed, the conditional variance of the estimates; it does not guarantee a larger estimation error in each realized sample. A larger $n$ makes $X'X$ grow roughly in proportion to $n$, so more errors are averaged out and the estimates move less. OLS is unbiased in all three designs, and its precision improves with a smaller error variance and a larger sample.

## 10. Reproduce the precision comparison

The sample SD uses `ddof=1`. The large-sample approximation is
$\operatorname{Var}(\hat\beta) \approx (V/n)Q^{-1}$, where
$Q=E[(1,x)'(1,x)]=\left[\begin{smallmatrix}1&1/2\\1/2&1/3\end{smallmatrix}\right]$
and $Q^{-1}=\left[\begin{smallmatrix}4&-6\\-6&12\end{smallmatrix}\right]$.
Thus the approximate SDs are $\sqrt{4V/n}$ and $\sqrt{12V/n}$.
These are not exact finite-sample unconditional SDs: in general,
$E[(X'X)^{-1}] \ne (E[X'X])^{-1}$.


In [9]:
print("DGP     empirical SD (b0, b1)     approximate SD (b0, b1)")
for name, estimates, n, V in [
    ("DGP1", dgp1, 20, 0.16),
    ("DGP2", dgp2, 20, 1.0),
    ("DGP3", dgp3, 200, 1.0),
]:
    print(name, estimates.std(axis=0, ddof=1), np.sqrt(V / n * np.array([4, 12])))


DGP     empirical SD (b0, b1)     approximate SD (b0, b1)
DGP1 [0.1616 0.2421] [0.1789 0.3098]
DGP2 [0.3584 0.5333] [0.4472 0.7746]
DGP3 [0.234  0.3347] [0.1414 0.2449]


## 11. Optional larger Monte Carlo check

The assignment requires ten runs; the following supplementary check uses 5,000 per DGP. A separate seed keeps the required results unchanged and makes this cell reproducible when rerun. These outputs should be used when updating the report's supplementary numbers; the report's existing 5,000-run numbers do not have a documented seed.


In [10]:
check_rng = np.random.default_rng(2027)
for name, n, V in [("DGP1", 20, 0.16), ("DGP2", 20, 1.0), ("DGP3", 200, 1.0)]:
    estimates = np.empty((5000, 2))
    for run in range(len(estimates)):
        y_check, x_check, _ = simulate(n, V, generator=check_rng)
        _, beta_check = ols(y_check, x_check)
        estimates[run] = beta_check.ravel()
    print(name, "mean:", estimates.mean(axis=0), "SD:", estimates.std(axis=0, ddof=1))


DGP1 mean: [ 4.9982 -2.4988] SD: [0.1881 0.3271]
DGP2 mean: [ 4.9989 -2.4898] SD: [0.4686 0.8245]
DGP3 mean: [ 4.9985 -2.4978] SD: [0.143  0.2465]
